# Timeseries metrics queries
## Introduction
This notebook illustrates using the WhyLabs timeseries metrics API to query profile and monitor metrics. Most of the examples use the REST API. Towards the end, there's some examples using the [whylabs-python-client](https://github.com/whylabs/whylabs-client-python). 

## Using the REST API


In [1]:
import requests
import os
import getpass
import json

In [2]:
base_url = "https://api.whylabsapp.com"
org_id = "org-0"
dataset_id = "model-0"

In [3]:
api_key = getpass.getpass()

········


In [4]:
headers = {"Accept": "application/json", "Content-Type": "application/json", "X-API-KEY": api_key}
url = base_url + f"/v0/organizations/{org_id}/dataset/{dataset_id}/data/metric-timeseries"

Get 5 days data from 19th March for the `bc_util` column and `quantile_95` metric. Queries are limited to at most 90 days.

In [5]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-03-19T00:00:00Z/P5D",
      "column": "bc_util",
      "metric": "quantile_95"
    }),
    headers=headers
)
resp.json()

{'data': [{'timestamp': 1710806400000,
   'lastModified': 1710882333280,
   'value': 469.6818354382451},
  {'timestamp': 1710892800000,
   'lastModified': 1710968747758,
   'value': 482.79219419934026},
  {'timestamp': 1710979200000,
   'lastModified': 1711055104866,
   'value': 476.2878226406801},
  {'timestamp': 1711065600000,
   'lastModified': 1711141547080,
   'value': 489.0629875981804},
  {'timestamp': 1711152000000,
   'lastModified': 1711227917918,
   'value': 207.7292131334232}]}

Get precision for a specified timerange.

In [6]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-03-19T00:00:00Z/P2D",
      "metric": "classification_precision"
    }),
    headers=headers
)
resp.json()

{'data': []}

Get a metric for a specific segment.

In [7]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-03-19T00:00:00Z/P2D",
      "column": "bc_util",
      "metric": "quantile_95",
      "segment": {
          "tags": [{"key": "verification_status", "value":"Source Verified"}]
      }
    }),
    headers=headers
)
resp.json()

{'data': [{'timestamp': 1710806400000,
   'lastModified': 1710882333248,
   'value': 476.5492592565752},
  {'timestamp': 1710892800000,
   'lastModified': 1710968747758,
   'value': 483.42130654590056}]}

Get data with an hourly granularity.

In [8]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-03-19T00:00:00Z/P2D",
      "column": "bc_util",
      "metric": "quantile_95",
      "granularity": "HOURLY"
    }),
    headers=headers
)
resp.json()

{'data': [{'timestamp': 1710882000000,
   'lastModified': 1710882333280,
   'value': 469.6818354382451},
  {'timestamp': 1710968400000,
   'lastModified': 1710968747758,
   'value': 482.79219419934026}]}

Get a rollup of a metric across the whole specified range.

In [9]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-03-19T00:00:00Z/P5D",
      "column": "bc_util",
      "metric": "quantile_95",
      "granularity": "ALL"
    }),
    headers=headers
)
resp.json()

{'data': [{'timestamp': 1710806400000,
   'lastModified': 1711227917918,
   'value': 413.9951103488429}]}

Get monitor metric `avg_drift` for a specific analyzer.

In [10]:
resp = requests.post(
    url=url,
    data=json.dumps({
      "interval": "2024-11-01T00:00:00Z/P1D",
      "column": "url",
      "metric": "avg_drift",
      "analyzerId": "comfortable-orchid-stinkbug-6423-analyzer"
    }),
    headers=headers
)
resp.json()

{'data': [{'timestamp': 1730419200000, 'value': 0.9329209663795195}]}

In [11]:
#!pip install "whylabs-client~=0.6.12"

In [12]:
import whylabs_client
from whylabs_client.api import data_api
from whylabs_client.model.metric_timeseries_request import MetricTimeseriesRequest
configuration = whylabs_client.Configuration(
    host = base_url
)
configuration.api_key['ApiKeyAuth'] = api_key

data_api = data_api.DataApi(whylabs_client.ApiClient(configuration))

Get data from 19th March for the `bc_util` column and `quantile_95` metric.

In [13]:
results = data_api.metric_timeseries_data(org_id, dataset_id, MetricTimeseriesRequest(
      interval = "2024-03-19T00:00:00Z/P1D",
      column = "bc_util",
      metric = "quantile_95",
))
results

{'data': [{'last_modified': 1710882333280,
           'timestamp': 1710806400000,
           'value': 469.6818354382451}]}

Get monitor metric `anomaly_count` for `bc_util` column and a specific analyzer. You can also omit the analyzer to get the total anomaly count for a column. Note that the python client uses snake case for field names like `monitor_id`, whereas the REST API uses lower camel case e.g. `monitorId`.

In [14]:
results = data_api.metric_timeseries_data(org_id, dataset_id, MetricTimeseriesRequest(
      interval = "2024-11-01T00:00:00Z/P1D",
      column = "url",
      metric = "avg_drift",
      monitor_id = "comfortable-orchid-stinkbug-6423",
))
results

{'data': [{'timestamp': 1730419200000, 'value': 0.9329209663795195}]}